# Generating Documentation Images for GlassCut

This notebook uses GlassCut itself on real sample data (`heart_tissue`, `breast_tissue`)
to generate figures for the documentation in `docs/_static/img/`.

## Setup

In [29]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from glasscut import Slide, GridTiler
from glasscut.data import ovarian_tissue
from glasscut.tissue_detectors import OtsuTissueDetector
from glasscut.stain_normalizers import MacenkoStainNormalizer

OUT = Path('../../docs/_static/img')
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'monospace',
    'font.size': 10,
    'axes.titlesize': 12,
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor': '#1e1e2e',
    'axes.edgecolor': '#313244',
    'axes.labelcolor': '#cdd6f4',
    'text.color': '#cdd6f4',
    'xtick.color': '#6c7086',
    'ytick.color': '#6c7086',
})

## Download sample data

Download real WSI samples (cached locally by `pooch`).

In [30]:
_, ovarian_path = ovarian_tissue()
print(f"Ovarian tissue: {ovarian_path}")

Ovarian tissue: /home/camilosinning/.cache/glasscut-data/tcga/ovarian/TCGA-13-1404-01A-01-TS1.cecf7044-1d29-4d14-b137-821f8d48881e.svs


## 1. Whole Slide Image Visualization

Open the heart tissue slide and show thumbnail + regions at different magnifications.

In [31]:
slide = Slide(str(ovarian_path))
print(f"Slide: {slide.name}")
print(f"Dimensions: {slide.dimensions}")
print(f"Magnifications: {slide.magnifications}")
print(f"MPP: {slide.mpp:.4f}")

Slide: TCGA-13-1404-01A-01-TS1.cecf7044-1d29-4d14-b137-821f8d48881e
Dimensions: (30001, 33987)
Magnifications: [20.0, 10.0, 5.0]
MPP: 0.5015


In [34]:
import PIL.Image

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

axes[0].imshow(slide.thumbnail)
axes[0].set_title('Thumbnail')

try:
    region_10 = slide.extract_tile((slide.dimensions[0] // 2, slide.dimensions[1] // 2), (1024, 1024), 10)
    axes[1].imshow(region_10.image)
    axes[1].set_title('Region @ 10x')
except Exception:
    axes[1].text(0.5, 0.5, 'N/A', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Region @ 10x')

try:
    mags = sorted(slide.magnifications, reverse=True)
    highest = mags[0]
    region_high = slide.extract_tile((slide.dimensions[0] // 2, slide.dimensions[1] // 2), (1024, 1024), highest)
    axes[2].imshow(region_high.image)
    axes[2].set_title(f'Region @ {highest}x')
except Exception as e:
    axes[2].text(0.5, 0.5, 'N/A', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title(f'Region @ high')

fig.tight_layout()
fig.savefig(OUT / 'slide_visualization.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved slide_visualization.png')

Saved slide_visualization.png


## 2. Grid Tiling Overlay

Show a grid overlay on the slide thumbnail, colouring tiles that pass tissue prescreening.

In [35]:
tiler = GridTiler(tile_size=(512, 512), magnification=20, min_tissue_ratio=0.1)
grid_overlay = tiler.visualize(slide)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(grid_overlay)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Grid tiles with tissue prescreening')
fig.savefig(OUT / 'grid_tiling.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved grid_tiling.png')

Saved grid_tiling.png


## 3. Tissue Detection

Show Otsu tissue detection on a slide region.

In [44]:
detector = OtsuTissueDetector()
first_mag = sorted(slide.magnifications, reverse=True)[0]
tile = slide.extract_tile((slide.dimensions[0] // 7, slide.dimensions[1] // 8), (1024, 1024), first_mag)
mask = detector.detect(tile.image)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
axes[0].imshow(tile.image)
axes[0].set_title('Original region')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Otsu tissue mask')
fig.tight_layout()
fig.savefig(OUT / 'tissue_detection.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved tissue_detection.png')

Saved tissue_detection.png


## 4. Stain Normalization

Show Macenko normalization: target stain from breast tissue, transform a heart region.

In [48]:
from glasscut.data import breast_tissue

_, breast_path = breast_tissue()
breast_slide = Slide(str(breast_path))
breast_mag = sorted(breast_slide.magnifications, reverse=True)[0]
target_tile = breast_slide.extract_tile((4*breast_slide.dimensions[0] // 5, breast_slide.dimensions[1] // 2), (1024, 1024), breast_mag)
target_img = target_tile.image
breast_slide.close()

normalizer = MacenkoStainNormalizer()
normalizer.fit(target_img)
source_img = tile.image
normalized = normalizer.transform(source_img)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
axes[0].imshow(source_img)
axes[0].set_title('Source (heart)')
axes[1].imshow(target_img)
axes[1].set_title('Target (breast)')
axes[2].imshow(normalized)
axes[2].set_title('Normalized')
fig.tight_layout()
fig.savefig(OUT / 'stain_normalization.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved stain_normalization.png')

Saved stain_normalization.png


## 5. Dataset Pipeline Diagram

A clean flow diagram showing the dataset generation pipeline.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off'); ax.set_facecolor('#1e1e2e')
ax.set_xlim(0, 10); ax.set_ylim(0, 3)
labels = ['Slides\n*.svs', 'GridTiler\ntiles', 'Tissue\nFilter', 'Dataset\nstorage']
positions = [(0.5, 1.0), (2.8, 1.0), (5.1, 1.0), (7.4, 1.0)]
for (x, y), label in zip(positions, labels):
    ax.add_patch(Rectangle((x, y), 1.6, 1.0, facecolor='#313244',
        edgecolor='#6366f1', lw=1.5))
    ax.text(x + 0.8, y + 0.5, label, ha='center', va='center',
        color='#cdd6f4', fontsize=9)
for i in range(len(positions) - 1):
    x1 = positions[i][0] + 1.6
    x2 = positions[i+1][0]
    ax.annotate('', xy=(x2, 1.5), xytext=(x1, 1.5),
        arrowprops=dict(arrowstyle='->', color='#6366f1', lw=2))
ax.set_title('Dataset generation pipeline', fontsize=11, color='#cdd6f4', pad=10)
fig.savefig(OUT / 'dataset_pipeline.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved dataset_pipeline.png')

## Cleanup

In [49]:
slide.close()
print('Done. Images saved to docs/_static/img/')

Done. Images saved to docs/_static/img/
